# Task 1 — Menyiapkan Evaluasi

Notebook ini menetapkan dan memvalidasi protokol evaluasi untuk sistem rekomendasi **Top-N**.

Keberhasilan Task 1 tidak ditentukan oleh nilai Recall@10 atau NDCG@10 tertentu, tetapi oleh implementasi evaluasi yang benar dan konsisten untuk semua model.

Aturan utama:
- menggunakan official training dan warm validation split;
- menggunakan evaluable users resmi;
- menggunakan candidate universe resmi;
- rating `>= 4` dianggap relevan;
- evaluasi dilakukan pada Top-10;
- item yang sudah terlihat saat training tidak boleh menjadi target atau rekomendasi;
- Final Test tidak digunakan untuk fitting, tuning, atau evaluasi notebook ini.

## 1. Setup proyek dan import

Notebook dapat dijalankan dari root proyek atau subfolder di dalam proyek.

In [1]:
from pathlib import Path
import json
import math
import sys

import numpy as np
import pandas as pd

# Cari root repository yang memiliki src/data.py.
PROJECT_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "src" / "data.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Root proyek dengan src/data.py tidak ditemukan.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data import load_project_data

K = 10
POSITIVE_THRESHOLD = 4.0
print("Project root:", PROJECT_ROOT)

Project root: d:\Folder Kuliah\SEMESTER 7\SISREK\Project\Hybrid-Fusion-Strategies-for-Collaborative-and-Content-Based-Recommender-Systems


## 2. Muat data resmi

`load_project_data()` mengambil training, metadata item, candidate universe warm, warm validation, dan daftar evaluable users. Tidak ada random split atau split baru yang dibuat.

In [2]:
data = load_project_data(PROJECT_ROOT)
train = data.train.copy()
items = data.items.copy()
validation_warm = data.validation_warm.copy()
candidates = list(dict.fromkeys(str(item_id) for item_id in data.candidates))
eval_users = list(dict.fromkeys(str(user_id) for user_id in data.evaluable_users))

for frame in (train, validation_warm):
    frame["user_id"] = frame["user_id"].astype(str)
    frame["item_id"] = frame["item_id"].astype(str)
items["item_id"] = items["item_id"].astype(str)

candidate_set = set(candidates)
eval_user_set = set(eval_users)
train_user_set = set(train["user_id"])
train_item_set = set(train["item_id"])
metadata_item_set = set(items["item_id"])

training_user_count = train["user_id"].nunique()
training_item_count = train["item_id"].nunique()
print(f"Training interactions : {len(train):,}")
print(f"Training users        : {training_user_count:,}")
print(f"Training items        : {training_item_count:,}")
print(f"Warm validation rows  : {len(validation_warm):,}")
print(f"Evaluable users       : {len(eval_users):,}")
print(f"Candidate universe    : {len(candidates):,}")

Training interactions : 427,957
Training users        : 54,483
Training items        : 22,753
Warm validation rows  : 27,537
Evaluable users       : 11,659
Candidate universe    : 22,753


## 3. Audit evaluable users dan candidate universe

Semua model harus dievaluasi pada user yang sama dan menggunakan ruang kandidat yang sama.

In [3]:
assert eval_user_set <= train_user_set, (
    f"Ada {len(eval_user_set - train_user_set):,} evaluable users tanpa histori training."
)
assert len(candidates) == len(candidate_set), "Candidate universe mengandung duplikat."
assert candidate_set <= train_item_set, "Ada kandidat yang tidak muncul di training."
assert candidate_set <= metadata_item_set, "Ada kandidat tanpa metadata item."

seen_by_user = {
    user_id: set(group["item_id"])
    for user_id, group in train.groupby("user_id", sort=False)
}

print("Audit evaluable users       : PASS")
print("Audit candidate universe    : PASS")
print("Candidate items in metadata: PASS")
print("Candidate items in training : PASS")

Audit evaluable users       : PASS
Audit candidate universe    : PASS
Candidate items in metadata: PASS
Candidate items in training : PASS


## 4. Bentuk ground truth validation

Item relevan harus memenuhi seluruh aturan berikut:
- user terdaftar sebagai evaluable user;
- rating validation `>= 4`;
- item berada di candidate universe;
- item belum pernah dilihat user tersebut pada training.

Semua evaluable users dipertahankan sebagai denominator macro-average, termasuk user tanpa target. Notebook akan mengaudit agar jumlah user tanpa target diketahui secara eksplisit.

In [4]:
validation_eval = validation_warm[
    validation_warm["user_id"].isin(eval_user_set)
].copy()

positive_validation = validation_eval.loc[
    (validation_eval["rating"] >= POSITIVE_THRESHOLD)
    & validation_eval["item_id"].isin(candidate_set)
].copy()

positive_validation["training_seen"] = [
    item_id in seen_by_user.get(user_id, set())
    for user_id, item_id in zip(
        positive_validation["user_id"],
        positive_validation["item_id"],
    )
]

relevant_rows = positive_validation.loc[
    ~positive_validation["training_seen"]
].copy()
relevant_by_user = {
    user_id: set(group["item_id"])
    for user_id, group in relevant_rows.groupby("user_id", sort=False)
}
for user_id in eval_users:
    relevant_by_user.setdefault(user_id, set())

users_without_target = [
    user_id for user_id in eval_users
    if not relevant_by_user[user_id]
]

print(f"Validation rows after user filter       : {len(validation_eval):,}")
print(f"Positive candidate validation rows      : {len(positive_validation):,}")
seen_target_count = int(positive_validation["training_seen"].sum())
print(f"Training-seen positive targets excluded : {seen_target_count:,}")
print(f"Relevant target rows                    : {len(relevant_rows):,}")
print(f"Users without relevant target           : {len(users_without_target):,}")

assert set(relevant_by_user) == eval_user_set
assert all(targets <= candidate_set for targets in relevant_by_user.values())
assert all(not (targets & seen_by_user.get(user_id, set()))
           for user_id, targets in relevant_by_user.items())

Validation rows after user filter       : 27,537
Positive candidate validation rows      : 24,202
Training-seen positive targets excluded : 0
Relevant target rows                    : 24,202
Users without relevant target           : 0


## 5. Implementasi dan sanity check metrik

Recall@10 mengukur proporsi target relevan yang ditemukan. NDCG@10 memberi bobot lebih besar pada target relevan yang berada di posisi atas. Keduanya dihitung sebagai macro-average antar-user.

In [5]:
def recall_at_k(ranked_items, relevant_items, k=K):
    ranked = [str(item_id) for item_id in ranked_items[:k]]
    targets = set(map(str, relevant_items))
    if not targets:
        return 0.0
    if len(ranked) != len(set(ranked)):
        raise ValueError("Ranking mengandung item duplikat.")
    return len(set(ranked) & targets) / len(targets)

def ndcg_at_k(ranked_items, relevant_items, k=K):
    ranked = [str(item_id) for item_id in ranked_items[:k]]
    targets = set(map(str, relevant_items))
    if not targets:
        return 0.0
    if len(ranked) != len(set(ranked)):
        raise ValueError("Ranking mengandung item duplikat.")
    dcg = sum(
        1.0 / math.log2(position + 2)
        for position, item_id in enumerate(ranked)
        if item_id in targets
    )
    ideal_count = min(k, len(targets))
    idcg = sum(
        1.0 / math.log2(position + 2)
        for position in range(ideal_count)
    )
    return dcg / idcg if idcg else 0.0

# Sanity checks matematis.
truth = {"A", "B"}
assert recall_at_k(["A", "B", "X"], truth) == 1.0
assert math.isclose(ndcg_at_k(["A", "B", "X"], truth), 1.0)
assert recall_at_k(["X", "A", "B"], truth) == 1.0
assert ndcg_at_k(["X", "A", "B"], truth) < 1.0
assert recall_at_k(["X", "Y"], truth) == 0.0
assert ndcg_at_k(["X", "Y"], truth) == 0.0
print("Sanity checks Recall@10/NDCG@10: PASS")

Sanity checks Recall@10/NDCG@10: PASS


## 6. Fungsi evaluasi ranking yang konsisten

Fungsi berikut dapat menerima keluaran model dalam format `[(item_id, score), ...]` atau `[item_id, ...]`. Fungsi juga mengaudit user universe, candidate restriction, seen-item exclusion, duplikasi, dan jumlah rekomendasi sebelum menghitung metrik.

In [6]:
def extract_item_ids(ranking):
    result = []
    for entry in ranking:
        result.append(str(entry[0]) if isinstance(entry, (tuple, list)) else str(entry))
    return result

def evaluate_recommendations(recommendations, model_name="model", k=K):
    recommendation_users = set(map(str, recommendations))
    if recommendation_users != eval_user_set:
        raise AssertionError(
            f"{model_name}: user universe berbeda dari evaluable users."
        )

    rows = []
    for user_id in eval_users:
        ranked = extract_item_ids(recommendations[user_id])
        if len(ranked) > k:
            raise AssertionError(f"{model_name}: lebih dari Top-{k} item.")
        if len(ranked) != len(set(ranked)):
            raise AssertionError(f"{model_name}: ranking memiliki duplikasi.")
        if not set(ranked) <= candidate_set:
            raise AssertionError(f"{model_name}: ada item di luar candidate universe.")
        if any(item_id in seen_by_user.get(user_id, set()) for item_id in ranked):
            raise AssertionError(f"{model_name}: ada item yang sudah dilihat user.")

        targets = relevant_by_user[user_id]
        rows.append({
            "model": model_name,
            "user_id": user_id,
            "n_relevant": len(targets),
            "n_recommended": len(ranked),
            f"recall@{k}": recall_at_k(ranked, targets, k),
            f"ndcg@{k}": ndcg_at_k(ranked, targets, k),
        })

    per_user = pd.DataFrame(rows)
    summary = {
        "model": model_name,
        "users_evaluated": len(per_user),
        f"recall@{k}": float(per_user[f"recall@{k}"].mean()),
        f"ndcg@{k}": float(per_user[f"ndcg@{k}"].mean()),
    }
    return summary, per_user

## 7. Smoke test dengan Popularity

Popularity hanya digunakan untuk memastikan pipeline evaluasi dapat dipakai oleh model. Nilainya bukan target keberhasilan Task 1.

In [7]:
positive_train = train.loc[train["rating"] >= POSITIVE_THRESHOLD]
popular_counts = positive_train.groupby("item_id").size().to_dict()
global_ranking = sorted(
    candidate_set,
    key=lambda item_id: (-popular_counts.get(item_id, 0), item_id),
)

popularity_recommendations = {}
for user_id in eval_users:
    seen = seen_by_user.get(user_id, set())
    popularity_recommendations[user_id] = [
        (item_id, float(popular_counts.get(item_id, 0)))
        for item_id in global_ranking
        if item_id not in seen
    ][:K]

popularity_summary, popularity_per_user = evaluate_recommendations(
    popularity_recommendations,
    model_name="Popularity smoke test",
    k=K,
)
pd.DataFrame([popularity_summary])

,model,users_evaluated,recall@10,ndcg@10
0,Popularity smoke test,11659,0.013499,0.009222


## 8. Ringkasan dan artefak Task 1

Ringkasan ini menunjukkan bahwa protokol sudah siap dipakai oleh Popularity, Item-CF, CBF, MF, dan Hybrid. Notebook tidak membaca atau memakai Final Test.

In [8]:
main_data_dirs = list(PROJECT_ROOT.glob("MAIN DATA-*/MAIN DATA"))
test_files = sorted(
    path.relative_to(PROJECT_ROOT).as_posix()
    for main_data_dir in main_data_dirs
    for path in main_data_dir.glob("interactions_test*.csv")
)

protocol_summary = {
    "task": "Task 1 - Evaluation Setup",
    "dataset": "Amazon Reviews 2023 - Musical Instruments",
    "train_interactions": int(len(train)),
    "validation_warm_rows_after_user_filter": int(len(validation_eval)),
    "positive_validation_candidate_rows": int(len(positive_validation)),
    "training_seen_targets_excluded": int(positive_validation["training_seen"].sum()),
    "relevant_target_rows": int(len(relevant_rows)),
    "candidate_universe_size": int(len(candidate_set)),
    "evaluable_users": int(len(eval_user_set)),
    "users_without_relevant_target": int(len(users_without_target)),
    "positive_relevant_threshold": POSITIVE_THRESHOLD,
    "metric_k": K,
    "test_files_detected": test_files,
    "uses_final_test": False,
    "protocol_audit": "PASS",
    "metric_sanity_checks": "PASS",
    "popularity_smoke_test": popularity_summary,
}

output_dir = PROJECT_ROOT / "outputs" / "task1"
output_dir.mkdir(parents=True, exist_ok=True)
(output_dir / "task1_evaluation_protocol_summary.json").write_text(
    json.dumps(protocol_summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
popularity_per_user.to_csv(
    output_dir / "task1_popularity_smoke_test_per_user.csv",
    index=False,
)

display(pd.DataFrame([protocol_summary]).T.rename(columns={0: "value"}))
print(f"\nTask 1 selesai. Artefak disimpan di: {output_dir}")

,value
task,Task 1 - Evaluation Setup
dataset,Amazon Reviews 2023 - Musical Instruments
train_interactions,427957
validation_warm_rows_after_user_filter,27537
positive_validation_candidate_rows,24202
training_seen_targets_excluded,0
relevant_target_rows,24202
candidate_universe_size,22753
evaluable_users,11659
users_without_relevant_target,0



Task 1 selesai. Artefak disimpan di: d:\Folder Kuliah\SEMESTER 7\SISREK\Project\Hybrid-Fusion-Strategies-for-Collaborative-and-Content-Based-Recommender-Systems\outputs\task1
